<a href="https://colab.research.google.com/github/Deepan32/deep-learning/blob/main/RNN_AK.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## RNN related imports

In [1]:

import numpy as np


In [3]:

# data I/O
data = open('/content/sample_data/input.txt', 'r').read() # should be simple plain text file
chars = list(set(data))

data_size, vocab_size = len(data), len(chars)


In [4]:
print('data has %d characters, %d unique.' % (data_size, vocab_size))

data has 175 characters, 24 unique.


In [5]:
char_to_ix = { ch:i for i,ch in enumerate(chars) }
ix_to_char = { i:ch for i,ch in enumerate(chars) }

In [6]:
# hyperparameters
hidden_size = 100 # size of hidden layer of neurons
seq_length = 24 # number of steps to unroll the RNN for
learning_rate = 1e-1

In [10]:
# model parameters

Wxh = np.random.randn(hidden_size, vocab_size)*0.01 # input to hidden
Whh = np.random.randn(hidden_size, hidden_size)*0.01 # hidden to hidden
Why = np.random.randn(vocab_size, hidden_size)*0.01 # hidden to output

In [11]:
# bias (also model parameters)
bh = np.zeros((hidden_size, 1))
by = np.zeros((vocab_size, 1)) # output bias

In [ ]:
def lossFun(inputs, targets, hprev):
  """
  inputs,targets are both list of integers.
  hprev is Hx1 array of initial hidden state
  returns the loss, gradients on model parameters, and last hidden state
  """
  xs, hs, ys, ps = {}, {}, {}, {}
  hs[-1] = np.copy(hprev)
  loss = 0
  # forward pass
  for t in xrange(len(inputs)):
    xs[t] = np.zeros((vocab_size,1)) # encode in 1-of-k representation
    xs[t][inputs[t]] = 1
    hs[t] = np.tanh(np.dot(Wxh, xs[t]) + np.dot(Whh, hs[t-1]) + bh) # hidden state
    ys[t] = np.dot(Why, hs[t]) + by # unnormalized log probabilities for next chars
    ps[t] = np.exp(ys[t]) / np.sum(np.exp(ys[t])) # probabilities for next chars
    loss += -np.log(ps[t][targets[t],0]) # softmax (cross-entropy loss)
  # backward pass: compute gradients going backwards
  dWxh, dWhh, dWhy = np.zeros_like(Wxh), np.zeros_like(Whh), np.zeros_like(Why)
  dbh, dby = np.zeros_like(bh), np.zeros_like(by)
  dhnext = np.zeros_like(hs[0])
  for t in reversed(xrange(len(inputs))):
    dy = np.copy(ps[t])
    dy[targets[t]] -= 1 # backprop into y. see http://cs231n.github.io/neural-networks-case-study/#grad if confused here
    dWhy += np.dot(dy, hs[t].T)
    dby += dy
    dh = np.dot(Why.T, dy) + dhnext # backprop into h
    dhraw = (1 - hs[t] * hs[t]) * dh # backprop through tanh nonlinearity
    dbh += dhraw
    dWxh += np.dot(dhraw, xs[t].T)
    dWhh += np.dot(dhraw, hs[t-1].T)
    dhnext = np.dot(Whh.T, dhraw)
  for dparam in [dWxh, dWhh, dWhy, dbh, dby]:
    np.clip(dparam, -5, 5, out=dparam) # clip to mitigate exploding gradients
  return loss, dWxh, dWhh, dWhy, dbh, dby, hs[len(inputs)-1]

In [8]:
Wxh

array([[-0.0200055 , -0.00018734,  0.00442025, ..., -0.0026492 ,
        -0.00057848, -0.00736644],
       [-0.00273789, -0.00842605,  0.00030929, ..., -0.01216516,
         0.00616914, -0.0118177 ],
       [-0.01064197, -0.00852278, -0.00984866, ...,  0.00874199,
         0.00596815,  0.01105414],
       ...,
       [-0.00295532,  0.00828141,  0.00412174, ...,  0.01258946,
         0.00057365, -0.01058503],
       [ 0.00776894,  0.00053699,  0.00271737, ...,  0.00118291,
         0.01819419,  0.00350654],
       [ 0.02365204, -0.0157897 ,  0.00719364, ..., -0.00834762,
        -0.00623359,  0.00440148]])

In [9]:
np.random.randn(1,2)

array([[-1.15297512, -0.6853484 ]])